# Main Figure 2

Frozen drawing logic and saved derived inputs. Outputs: `outputs/main/figure_02`. See reproduction status before release.


In [ ]:
from pathlib import Path
import os, sys
sys.dont_write_bytecode = True
_start = Path.cwd().resolve()
REPO = next((p for q in [_start, *_start.parents] for p in [q, q / "github"]
             if (p / "plot_utils" / "release_runtime.py").is_file()), None)
if REPO is None:
    raise RuntimeError("Run from the repository root or a notebook subdirectory.")
sys.path.insert(0, str(REPO / "plot_utils"))
from release_runtime import configure
configure(REPO)


In [ ]:

from pathlib import Path
import os, sys, json, hashlib, tempfile
sys.dont_write_bytecode=True
PROJECT_ROOT=ROOT=REPO/'figure_source_data/shared'
OUTPUT_ROOT=REPO/'outputs/main/figure_02'
OUTPUT_ROOT.mkdir(parents=True,exist_ok=True)
RUNTIME=OUTPUT_ROOT/'runtime'; RUNTIME.mkdir(exist_ok=True)
FIG=ROOT/'fig'
for key in ['MPLCONFIGDIR','XDG_CACHE_HOME','TEMP','TMP','TMPDIR','NUMBA_CACHE_DIR']:
    os.environ[key]=str(RUNTIME)
tempfile.tempdir=str(RUNTIME)


In [ ]:
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D
from matplotlib.patches import Patch, Rectangle
from matplotlib.colors import LinearSegmentedColormap, TwoSlopeNorm, LogNorm, Normalize
from matplotlib.collections import LineCollection
from matplotlib.ticker import ScalarFormatter
from scipy.ndimage import gaussian_filter1d
import cartopy
import cartopy.crs as ccrs
from cartopy.io import shapereader
cartopy.config['cache_dir'] = RUNTIME / 'cartopy'
import geopandas as gpd


SOURCE_HASHES = {
    PROJECT_ROOT / 'fig/data_cache/v2/clock_soil_composite.csv': '9d3ae46d767b3ad4d15c41422b28e99e8696e6251fb005e107bed78580ed58a7',
    PROJECT_ROOT / 'fig/Figure2/data_cache/clock_density.csv': 'a12c9c82b8039026df026a3627b5fec2f32b7abfc3684401b458674c5bfab5a6',
    PROJECT_ROOT / 'fig/Figure2/data_cache/clock_summary.csv': 'd84aacabba1e3c0e7cfdf9fd825907021089d6dd5f83cd35e95dd2fc0a54a56e',
    PROJECT_ROOT / 'fig/Figure2/data_cache/pair_density.npz': '8dde44305e1100457d2815bf9eb7cb99ff877a5c0ffb740612b4796f49ee8489',
    PROJECT_ROOT / 'fig/Figure2/data_cache/pair_metrics.csv': 'abdb662654f7d8c7fa5a2bfab8a825a42b7f11185f05e5d4c386994ce24ab381',
    PROJECT_ROOT / 'fig/Figure2/data_cache/soil_cdf_curves.csv': 'b7a145ab0c150eb04f59c2b8f2d6ddafcc957b282da359e2a2bc96c1c49ed23f',
    PROJECT_ROOT / 'fig/Figure2/data_cache/stability/diagnostic_event_metrics.parquet': 'd7d1a70cd774ff032e03dc9b3675483235b6e0af4fec3b17fc660f5f77df54a3',
    PROJECT_ROOT / 'fig/Figure2/data_cache/stability/jitter_amplitude.csv': '36262667a52c2e3df4e76de52df743076435effdc3dc173740234c77f8a01091',
    PROJECT_ROOT / 'fig/Figure2/data_cache/stability/jitter_by_structure.csv': '2c5736e3e886a85019ff4a1dd0851600b7dbc40fe073813fd331bbdf439bfc51',
}
for path, expected in SOURCE_HASHES.items():
    print(f'{path} | exists={path.is_file()}')
    assert path.is_file(), path
    assert hashlib.sha256(path.read_bytes()).hexdigest() == expected, f'Changed frozen input: {path}'
print('Reads archive: NO; reads excluded SRI directory: NO')


FROZEN_TABLES = {p: pd.read_csv(p) for p in SOURCE_HASHES if p.suffix == '.csv'}
for path, table in FROZEN_TABLES.items():
    print(path.name, 'rows=', len(table), 'columns=', list(table.columns))
def frozen_csv(path):
    path = Path(path).resolve()
    assert path in FROZEN_TABLES, f'Undeclared CSV input: {path}'
    return FROZEN_TABLES[path].copy()


summary=frozen_csv(PROJECT_ROOT/'fig/Figure2/data_cache/clock_summary.csv')
assert np.allclose(summary.iqr, [6,16,4.71047], atol=1e-5)
events=pd.read_parquet(PROJECT_ROOT/'fig/Figure2/data_cache/stability/diagnostic_event_metrics.parquet')
assert len(events)==189254
composite=frozen_csv(PROJECT_ROOT/'fig/data_cache/v2/clock_soil_composite.csv')
assert (composite['n']==196597).all()
print('Global reference timings: 3,135,834; soil cohort: 189,254; composite: 196,597; jitter: 187,977')


C = PROJECT_ROOT/'fig/Figure2/data_cache'
COL={'nearest':'#467EA2','peak':'#CC8255','com':'#238A83'};LAB={'nearest':'Nearest','peak':'Peak→onset','com':'CoM→onset'};LAY=['rootzone_sm_0_100','deep_sm_100_289','profile_sm_0_289'];LL=['Root zone','Deep','Profile']
plt.rcParams.update({'font.family':'Arial','font.size':7,'axes.labelsize':7,'xtick.labelsize':6,'ytick.labelsize':6,'axes.linewidth':.55,'axes.spines.top':False,'axes.spines.right':False,'lines.linewidth':1,'xtick.major.size':2.5,'ytick.major.size':2.5,'xtick.major.width':.5,'ytick.major.width':.5,'pdf.fonttype':42,'svg.fonttype':'none','text.color':'#303840','axes.labelcolor':'#303840','savefig.facecolor':'white'})
def read(s):return frozen_csv(C/s)
def title(fig,spec,l,t):
 b=spec.get_position(fig);fig.text(b.x0-.033,b.y1+.024,l,fontweight='bold',fontsize=9,va='bottom');fig.text(b.x0,b.y1+.024,t,fontweight='bold',fontsize=6.6,va='bottom')
def violin(ax,g,r,x,col,width=.33):
 g=g[(g.x>=r.minimum)&(g.x<=r.maximum)];v=gaussian_filter1d(g.density.to_numpy(),1.2);w=v/v.max()*width;ax.fill_betweenx(g.x,x-w,x+w,color=col,alpha=.68,lw=.45,edgecolor=col);ax.plot([x,x],[r.q25,r.q75],color='#303840',lw=2.1);ax.plot(x,r['median'],'o',ms=3,mfc='white',mec='#303840',mew=.55)
def grouped(ax,d):
 for j,k in enumerate(COL):
  z=d[d.clock==k].set_index('layer').loc[LAY];ax.errorbar(z['mean'],np.arange(3)+(j-1)*.20,xerr=[z['mean']-z.lo,z.hi-z['mean']],fmt='o',ms=3.2,color=COL[k],capsize=1.7,elinewidth=.8)
 ax.set_yticks(range(3),LL);ax.set_ylim(2.5,-.5)



plt.rcParams['font.family']='sans-serif'
plt.rcParams['font.sans-serif']=['Arial','DejaVu Sans','Liberation Sans']
plt.rcParams['svg.fonttype']='none'


In [ ]:

PRODUCT={'GRUN':'#007C78','ERA5':'#CB7935','GLDAS':'#667BA5'}
ORDER=list(PRODUCT);MARK={'nearest':'o','peak':'s','com':'D'}
P8=ROOT/'Part8_change_work';DIAG=ROOT/'result1.0/diagnostics/figure2_diagnostics'
def finalcsv(path):
    SOURCE_HASHES[path]=hashlib.sha256(path.read_bytes()).hexdigest()
    return pd.read_csv(path)
summary3=finalcsv(P8/'figure_source_data/Figure2_source_data_summary.csv')
pairs3=finalcsv(P8/'figure_source_data/Figure2_source_data_pairs.csv')
stability3=finalcsv(P8/'SI/SI04_tables/Figure2_source_data_cross_product_perturbation.csv')
stability_qc=finalcsv(DIAG/'Figure2_cross_product_perturbation_QC.csv')
assert set(summary3['product'])==set(ORDER) and len(summary3)==9
assert set(pairs3['product'])==set(ORDER) and len(pairs3)==6
assert set(stability3['product'])==set(ORDER) and len(stability3)==54
assert stability3.amplitude.eq(1).all() and stability3.repetitions.eq(200).all()
assert stability_qc.maximum_CoM_reproduction_error.lt(2e-4).all()
assert np.allclose(summary3.q75-summary3.q25,summary3.IQR)
for product,folder in [('GRUN','GRUN_try'),('ERA5','ERA5_try'),('GLDAS','GLADS_try')]:
    ns=summary3.loc[summary3['product']==product,'n'].unique();assert len(ns)==1
    assert pairs3.loc[pairs3['product']==product,'n'].eq(ns[0]).all()
    st=stability3[stability3['product']==product]
    for clock in COL:
        assert st[(st.clock==clock)&(st.parent_bin!='all')].n.sum()==st[(st.clock==clock)&(st.parent_bin=='all')].n.iloc[0]
    single=st[st.parent_bin=='1'];assert np.ptp(single.mean_absolute_change)<1e-12
print('a/c/e: FOUND EXISTING for GRUN, ERA5 and GLDAS. No new calculation.')


In [ ]:

plt.rcParams.update({'axes.labelsize':6,'xtick.labelsize':5.4,'ytick.labelsize':5.4})
fig=plt.figure(figsize=(7.20472440945,195/25.4))
gs=fig.add_gridspec(2,3,left=.078,right=.975,bottom=.14,top=.865,wspace=.52,hspace=.59)
fig.legend(handles=[Line2D([],[],color=COL[k],marker=MARK[k],ms=3,lw=1.0,label=LAB[k]) for k in COL],title='Reference colours: b / d / e · Reference symbols: a',title_fontsize=5.2,loc='upper center',bbox_to_anchor=(.52,.995),ncol=3,frameon=False,fontsize=6)
fig.legend(handles=[Line2D([],[],color=PRODUCT[p],marker='o',ms=3,lw=1,label=p) for p in ORDER],title='Runoff products: a / c / e',title_fontsize=5.2,loc='upper center',bbox_to_anchor=(.52,.952),ncol=3,frameon=False,fontsize=5.8)


In [ ]:

sub=gs[0,0].subgridspec(3,1,hspace=.52)
for j,product in enumerate(ORDER):
    ax=fig.add_subplot(sub[j]);d=summary3[summary3['product']==product].set_index('clock')
    for i,k in enumerate(COL):
        r=d.loc[k]
        ax.plot([r.q25,r.q75],[i,i],color=PRODUCT[product],lw=1.8 if product=='GRUN' else 1.3)
        ax.plot(r['median'],i,marker=MARK[k],ms=3.2,color=PRODUCT[product],mec='white',mew=.35)
    ax.set(ylim=(2.5,-.5),xlim=(0,24),yticks=range(3),yticklabels=['Nearest','Peak','CoM'],xticks=[0,12,24])
    ax.set_title(f'{product} · n={int(d.n.iloc[0]):,}',loc='left',fontsize=5.2,color=PRODUCT[product],pad=3)
    if j<2:ax.tick_params(labelbottom=False)
    else:ax.set_xlabel('Interval to H onset (months)',fontsize=5.5)
title(fig,gs[0,0],'a','Global reference distributions')
b=gs[0,0].get_position(fig);fig.text(b.x0,b.y0-.053,'Median and IQR · same-event sample',fontsize=5.1)


In [ ]:
ax=fig.add_subplot(gs[0,1]);data=frozen_csv(ROOT/'fig/data_cache/v2/clock_soil_composite.csv');data=data[(data.layer=='rootzone_sm_0_100')&(data.state=='true')]
for k in COL:
 g=data[data.clock==k];ax.fill_between(g.month,g.lo,g.hi,color=COL[k],alpha=.16,lw=0);ax.plot(g.month,g['mean'],color=COL[k],lw=1.1)
ax.axhline(0,color='#899299',ls='--',lw=.6);ax.axvline(0,color='#899299',ls=':',lw=.6);ax.set(xlim=(-12,12),xlabel='Months relative to meteorological anchor',ylabel='Root-zone soil anomaly (z)');ax.set_xticks([-12,-6,0,6,12]);ax.xaxis.label.set_size(5.3);ax.margins(y=.3);ax.text(.03,.97,f'{int(data.n.iloc[0]):,} events · 95% block CI',transform=ax.transAxes,va='top',fontsize=5.0);ax.text(.03,.035,'Negative = drier than monthly climatology',transform=ax.transAxes,fontsize=5.0);title(fig,gs[0,1],'b','ERA5 root-zone soil moisture')


In [ ]:

sub=gs[0,2].subgridspec(2,1,hspace=.72)
for j,(metric,label,limits) in enumerate([('rho','Spearman ρ',(0,1)),('median_absolute_difference','Median absolute difference (months)',(0,10))]):
    ax=fig.add_subplot(sub[j])
    for i,product in enumerate(ORDER):
        z=pairs3[pairs3['product']==product].set_index('clock')
        for clock,offset,marker in [('nearest',-.13,'o'),('peak',.13,'s')]:
            v=z.loc[clock,metric];ax.plot(v,i+offset,marker=marker,ms=3.6 if product=='GRUN' else 3,color=PRODUCT[product],mfc='white' if clock=='nearest' else PRODUCT[product],mew=.8)
    ax.set(yticks=range(3),yticklabels=ORDER,ylim=(2.5,-.5),xlim=limits,xlabel=label)
    ax.xaxis.label.set_size(5.0)
title(fig,gs[0,2],'c','Event-level reference relationships')
box=gs[0,2].get_position(fig)
fig.legend(handles=[Line2D([],[],ls='',marker='o',mfc='white',mec='#555',ms=3,label='Nearest vs CoM'),Line2D([],[],ls='',marker='s',color='#555',ms=3,label='Peak vs CoM')],loc='upper left',bbox_to_anchor=(box.x0-.01,box.y0-.05),frameon=False,fontsize=5.1,handletextpad=.4,ncol=1)


In [ ]:
curves=read('soil_cdf_curves.csv');sub=gs[1,0].subgridspec(3,1,hspace=.48)
for l,layer in enumerate(LAY):
 ax=fig.add_subplot(sub[l]);z=curves[curves.layer==layer]
 for k in COL:
  g=z[z.clock==k];ax.plot(g.threshold,g.observed_cdf,color=COL[k],lw=1.1);ax.plot(g.threshold,g.null_cdf,color=COL[k],ls='--',lw=.7,alpha=.72)
 ax.set_xscale('symlog',linthresh=24);ax.set(xlim=(0,128),ylim=(0,102));ax.set_xticks([0,12,24,96]);ax.xaxis.set_major_formatter(ScalarFormatter());ax.set_yticks([0,50,100]);ax.text(.02,1.04,LL[l],transform=ax.transAxes,fontsize=5.5,va='bottom')
 if l==1:ax.set_ylabel('Events below threshold (%)',fontsize=6)
 if l==2:ax.set_xlabel('Distance to SM-CoM (months)',fontsize=5.8)
 else:ax.tick_params(labelbottom=False)
title(fig,gs[1,0],'d','ERA5 soil-timing distributions')


In [ ]:

sub=gs[1,1].subgridspec(3,1,hspace=.52)
for j,product in enumerate(ORDER):
    ax=fig.add_subplot(sub[j]);z=stability3[(stability3['product']==product)&(stability3.parent_bin!='all')]
    for clock in COL:
        q=z[z.clock==clock].set_index('parent_bin').loc[['1','2','3','4','5+']]
        ax.plot(range(5),q.mean_absolute_change,color=COL[clock],marker=MARK[clock],ms=3.4 if clock=='nearest' else 2.3,mfc='white' if clock=='nearest' else COL[clock],ls='--' if clock=='peak' else '-',lw=.9)
    ne=int(stability_qc.loc[stability_qc['product']==product,'n_events'].iloc[0])
    ax.set_title(f'{product} · n={ne:,}',loc='left',fontsize=5.2,color=PRODUCT[product],pad=3)
    ax.set(ylim=(0,.56),xlim=(-.2,4.2),xticks=range(5),xticklabels=['1','2','3','4','5+'],yticks=[0,.25,.5])
    if j<2:ax.tick_params(labelbottom=False)
    else:ax.set_xlabel('Number of antecedents\n1: single · ≥2: multiple',fontsize=5.3)
    if j==1:ax.set_ylabel('Mean absolute change (months)',fontsize=5.3)
title(fig,gs[1,1],'e','Timing stability across structures')


In [ ]:
sub=gs[1,2].subgridspec(2,1,height_ratios=[1,.06],hspace=.52);ax=fig.add_subplot(sub[0]);s=pd.read_parquet(C/'stability/diagnostic_event_metrics.parquet');x=s.min_distance_peak;y=s.min_distance_com
im=ax.hexbin(x,y,gridsize=45,mincnt=1,norm=LogNorm(),cmap=LinearSegmentedColormap.from_list('density',['#F1F6F6','#8ABAB9','#205E65']),rasterized=True);mx=np.ceil(max(x.max(),y.max()));ax.plot([0,mx],[0,mx],'--',color='#626C72',lw=.7);ax.set(xlim=(0,mx),ylim=(0,mx),xlabel='Peak distance (months)',ylabel='CoM distance (months)');ax.set_xticks([0,20,40]);ax.set_yticks([0,20,40]);ax.text(.03,.95,'Below diagonal: CoM is closer',transform=ax.transAxes,va='top',fontsize=5.0);ax.text(.03,.85,'Reference: driest soil month',transform=ax.transAxes,va='top',fontsize=5.0)
cb=fig.colorbar(im,cax=fig.add_subplot(sub[1]),orientation='horizontal',ticks=[1,10,100,1000]);cb.outline.set_visible(False);cb.ax.tick_params(labelsize=5);cb.set_label('Events per hexagon (log)',fontsize=5.3,labelpad=1);title(fig,gs[1,2],'f','ERA5 same-event soil distances')


In [ ]:

FIGURE_NUMBER=2
FROZEN_TITLES={'a': 'Temporal references yield distinct timings', 'b': 'Soil-moisture trajectories differ across anchors', 'c': 'Event timings correspond imperfectly', 'd': 'CoM lies closer to soil timing', 'e': 'CoM timing stabilizes with antecedent multiplicity', 'f': 'Soil-timing distances differ within events'}
from matplotlib.text import Text
from matplotlib.colors import TwoSlopeNorm, LinearSegmentedColormap
import textwrap
OUTDIR=Path(OUT if FIGURE_NUMBER==6 else OUTPUT_ROOT)
QA=OUTDIR/'qa';QA.mkdir(exist_ok=True)
def numerical_artists():
    result={}
    def add(key,value):
        a=np.ma.asarray(value)
        if a.dtype.kind in 'biufc':
            v=np.ascontiguousarray(a.filled(np.nan) if np.ma.isMaskedArray(a) and a.dtype.kind in 'fc' else a)
            result[key]={'shape':list(v.shape),'sha256':hashlib.sha256(v.tobytes()).hexdigest()}
    for j,a in enumerate(fig.axes):
        for k,l in enumerate(a.lines):add(f'{j}/line{k}/x',l.get_xdata());add(f'{j}/line{k}/y',l.get_ydata())
        for k,l in enumerate(a.collections):
            if l.get_array() is not None:add(f'{j}/collection{k}/array',l.get_array())
            if hasattr(l,'get_offsets'):add(f'{j}/collection{k}/offsets',l.get_offsets())
            if hasattr(l,'get_segments'):
                for h,v in enumerate(l.get_segments()):add(f'{j}/collection{k}/segment{h}',v)
            if l.__class__.__name__ in ('PolyCollection','FillBetweenPolyCollection'):
                for h,v in enumerate(l.get_paths()):add(f'{j}/collection{k}/path{h}',v.vertices)
        for k,l in enumerate(a.images):add(f'{j}/image{k}',l.get_array())
        for k,l in enumerate(a.patches):
            if isinstance(l,matplotlib.patches.Rectangle):add(f'{j}/bar{k}',[l.get_x(),l.get_y(),l.get_width(),l.get_height()])
    return result
fig.canvas.draw()
before=numerical_artists()
contracts=json.loads((REPO/'assets/other/panel_contracts.json').read_text(encoding='utf-8'))
oldtitles={r[2].replace('clock','reference').replace('Clock','Reference') for r in contracts if r[0]==FIGURE_NUMBER}
notes=[]
for t in list(fig.findobj(Text)):
    s=t.get_text()
    if s in oldtitles or s in list('abcdef'):
        t.set_text('');continue
    if len(s)>180:
        notes.append(s);t.set_text('');continue
    s=s.replace('decade$^{-1}$','decade⁻¹').replace(r'$\beta$','β').replace('$R^2$','R²')
    t.set_text(s);t.set_fontfamily('DejaVu Sans');t.set_fontsize(5.5 if FIGURE_NUMBER in (1,2) else 6)
    if t.get_rotation():t.set_rotation_mode('anchor')
plt.rcParams.update({'font.family':'sans-serif','font.sans-serif':['DejaVu Sans','Arial','Liberation Sans'],
    'font.size':6,'axes.labelsize':6,'xtick.labelsize':5.5,'ytick.labelsize':5.5,
    'legend.fontsize':5.5,'pdf.fonttype':42,'svg.fonttype':'none'})
DIV=plt.get_cmap('RdBu_r')
for ax in fig.axes:
    cb=getattr(ax,'_colorbar',None)
    for sp in ax.spines.values():sp.set_linewidth(.45);sp.set_edgecolor('#657076')
    ax.tick_params(width=.45,length=2,pad=2,labelsize=5.5)
    ax.xaxis.label.set_fontsize(6);ax.yaxis.label.set_fontsize(6)
    ax.title.set_fontsize(6)
    if cb is not None:
        cb.outline.set_linewidth(.35)
        ax.tick_params(length=1.5,pad=1)
        ax.set_axes_locator(None)
        p=ax.get_position()
        if cb.orientation=='horizontal':ax.set_position([p.x0,p.y0,p.width,min(p.height,2.5/(72*fig.get_figheight()))])
        else:ax.set_position([p.x0,p.y0,min(p.width,2.5/(72*fig.get_figwidth())),p.height])
    for artist in list(ax.images)+list(ax.collections):
        norm=getattr(artist,'norm',None)
        if norm is not None and norm.vmin is not None and norm.vmax is not None and norm.vmin<0<norm.vmax:
            assert np.isclose(abs(norm.vmin),abs(norm.vmax)), 'Original diverging limits require review'
            artist.set_cmap(DIV);artist.set_norm(TwoSlopeNorm(vmin=norm.vmin,vcenter=0,vmax=norm.vmax))
    for line in ax.lines:
        if line.get_linestyle() in ('--',':'):
            x=np.asarray(line.get_xdata());y=np.asarray(line.get_ydata())
            if (len(x)>0 and np.all(x==0)) or (len(y)>0 and np.all(y==0)):
                line.set_color('#8B9298');line.set_linewidth(.5);line.set_linestyle('--')
        if line.get_markersize()>4:line.set_markersize(3.5)
        if line.get_linewidth()>1.8:line.set_linewidth(1.5)
    if hasattr(ax,'projection'):
        for col in ax.collections:
            if col.__class__.__name__=='FeatureArtist':col.set_linewidth(.25)
        ax.spines['geo'].set_linewidth(.45)

if FIGURE_NUMBER==1:
    for ax in fig.axes:
        for t in ax.texts:
            if t.get_text()=='24 mo':t.set_x(.62);t.set_transform(ax.transAxes);t.set_y(.07);t.set_ha('right')
            if t.get_text().startswith('d1  '):t.set_position((0,1.02));t.set_va('bottom')
            if t.get_text().startswith('Common 36-month'):
                notes.append(t.get_text());t.set_text('Common 36-month\ncoverage cohort');t.set_position((.04,.06));t.set_fontsize(5)
if FIGURE_NUMBER==2:
    for ax in fig.axes:
        if ax.get_xlabel()=='Months relative to meteorological anchor':ax.set_xlabel('Months from meteorological anchor',fontsize=6)
        for t in ax.texts:
            if t.get_text().startswith('Negative ='):notes.append(t.get_text());t.set_text('')
            if '95% block CI' in t.get_text():t.set_position((.03,1.02));t.set_va('bottom');t.set_fontsize(5)
    for ax in fig.axes:
        cb=getattr(ax,'_colorbar',None)
        if cb is not None and 'hexagon' in ax.get_xlabel():
            from matplotlib.ticker import FixedLocator,FixedFormatter,NullLocator
            ax.xaxis.set_major_locator(FixedLocator([1,10,100,1000]));ax.xaxis.set_major_formatter(FixedFormatter(['1','10','100','1000']));ax.xaxis.set_minor_locator(NullLocator())
if FIGURE_NUMBER==3:
    for leg in list(fig.legends):
        if any(t.get_text()=='Q10–Q90' for t in leg.get_texts()):
            handles=leg.legend_handles;labels=[t.get_text() for t in leg.get_texts()]
            pos=gs[0,0].get_position(fig);leg.remove()
            fig.legend(handles,labels,ncol=4,frameon=False,fontsize=5.2,loc='upper left',bbox_to_anchor=(pos.x0-.005,pos.y0-.038),handlelength=1.2,columnspacing=.8)
    maps=[a for a in fig.axes if hasattr(a,'projection')]
    for a in maps:
        a.title.set_fontsize(5.5)
    for t in fig.texts:
        if t.get_text().startswith('Main widths pool events'):
            t.set_text('Main widths: pooled events. d2/f: grid-year means. Grey: no estimate.');t.set_fontsize(5.5)
    for ax in fig.axes:
        for t in ax.texts:
            if t.get_text().startswith('Thin: annual'):
                t.set_position((.02,1.01));t.set_va('bottom');t.set_text('Thin: annual; thick: 5-year visual mean');t.set_fontsize(5)
        cb=getattr(ax,'_colorbar',None)
        if cb and cb.orientation=='horizontal':
            p=ax.get_position();ax.set_position([p.x0,p.y0-.01,p.width,p.height]);cb.set_ticks([cb.norm.vmin,0,cb.norm.vmax] if cb.norm.vmin<0 else [0,50,100])
        if ax.images:
            p=ax.get_position();ax.set_position([p.x0+.04,p.y0,p.width-.04,p.height]);ax.tick_params(axis='y',labelsize=5)
if FIGURE_NUMBER==4:
    for leg in fig.legends:leg.set_bbox_to_anchor((.5,.045))
    for ax in fig.axes:
        for t in ax.texts:
            s=t.get_text().replace('Parent number','Antecedent count').replace('Many-to-one','Multiple antecedents')
            if '  ρ: ' in s:
                s=s.replace('  ρ: ','\nρ: ');t.set_position((.01,1.03));t.set_va('bottom');t.set_fontsize(5.2)
            if s.startswith('a1  '):
                t.set_text('');p=ax.get_position();fig.text(p.x0,p.y1+.013,'a1  Event scale · GRUN',fontsize=5.5,fontweight='bold')
            else:t.set_text(s)
            if not ax.axison:t.set_fontsize(5.2)
        if ax.get_ylabel()=='TP slope':ax.set_ylabel('PT slope',fontsize=6)
        cb=getattr(ax,'_colorbar',None)
        if cb is not None:cb.set_ticks([-1,0,1])
if FIGURE_NUMBER==5:
    for ax in fig.axes:
        for t in ax.texts:
            s=t.get_text()
            if s.startswith('Envelope spans') or s.startswith('Filled:'):
                t.set_position((0,-.22));t.set_ha('left');t.set_va('top');t.set_fontsize(5.2)
            if s=='Severity decile':t.set_x(-.29);t.set_fontsize(5.5)
            if 'median β < 0' in s:
                t.set_position((.0,-.32));t.set_ha('left');t.set_va('top');t.set_text(s.replace('\n','; ',1));t.set_fontsize(5.2);t.set_bbox(None)
        if hasattr(ax,'projection'):
            leg=ax.get_legend()
            if leg:leg.set_bbox_to_anchor((0,-.10));leg._loc=2
            for cax in fig.axes:
                cb=getattr(cax,'_colorbar',None)
                if cb and cb.orientation=='horizontal':
                    p=ax.get_position();cax.set_position([p.x0,p.y0-.105,p.width,2.5/(72*fig.get_figheight())])
if FIGURE_NUMBER==6:
    for ax in fig.axes:
        for t in ax.texts:
            if t.get_text()=='threshold floor':t.set_text('AUC < 0.60: L* = 0');t.set_x(.34);t.set_fontsize(5.5)
            if t.get_text().startswith('Δ ='):
                t.set_x(1.0);t.set_fontsize(5);t.set_bbox(None)
                if ax.get_title()=='Retrospective PT':
                    t.set_x(.02);t.set_ha('left');t.set_text(t.get_text().replace('Δ = ','Δ '))
            if t.get_text() in ('GRUN','ERA5','GLDAS') and ax in fig.axes[12:]:
                t.set_position((.50,.84));t.set_fontsize(5.5)
    for t in fig.texts:
        if t.get_text().startswith('Points: temporal'):t.set_y(.025);t.set_fontsize(5)

if FIGURE_NUMBER in (1,2,3):specs=[gs[0,0],gs[0,1],gs[0,2],gs[1,0],gs[1,1],gs[1,2]]
elif FIGURE_NUMBER==4:specs=[gs[0,0],gs[0,1],gs[1,0],gs[1,1]]
elif FIGURE_NUMBER==5:specs=[outer[0,0],outer[0,1],outer[0,2],outer[1,0],outer[1,1],outer[1,2]]
else:specs=top+bottom
headers=[]
for letter,spec in zip('abcdef',specs):
    b=spec.get_position(fig);h=fig.get_figheight()*72;w=fig.get_figwidth()*72
    width_chars=33 if FIGURE_NUMBER in (1,2) else 48
    title_offset=9 if FIGURE_NUMBER in (1,2) else 24 if FIGURE_NUMBER==4 else 18
    txt=textwrap.fill(FROZEN_TITLES[letter],width_chars,break_long_words=False)
    t=fig.text(b.x0,b.y1+title_offset/h,txt,ha='left',va='bottom',fontsize=6.5,fontweight='normal',linespacing=1.4,color='#20272B')
    fig.text(b.x0-10/w,b.y1+title_offset/h,letter,ha='left',va='bottom',fontsize=8,fontweight='bold',color='#20272B')
    headers.append(t)
fig.canvas.draw()
after=numerical_artists()
assert before==after,'A scientific artist changed during styling'
(QA/'numerical_artist_invariance.json').write_text(json.dumps({'status':'PASS','entries':len(before),'before':before,'after':after},indent=2),encoding='utf-8')
(QA/'retained_caption_notes.json').write_text(json.dumps(notes,ensure_ascii=False,indent=2),encoding='utf-8')
from audit_panel_alignment import require_matplotlib_panel_alignment
eligible=[a for a in fig.axes if a.axison and getattr(a,'_colorbar',None) is None and not hasattr(a,'projection')]
if len(eligible)<2:eligible=[a for a in fig.axes if a.axison and getattr(a,'_colorbar',None) is None]
ids=[f'plot{i+1}' for i in range(len(eligible))]
rows={};cols={}
for ident,ax in zip(ids,eligible):
    p=ax.get_position();rows.setdefault((round(p.y0,5),round(p.y1,5)),[]).append(ident);cols.setdefault((round(p.x0,5),round(p.x1,5)),[]).append(ident)
rg=[v for v in rows.values() if len(v)>1];cg=[v for v in cols.values() if len(v)>1]
ex=[]
for group in cg:
    if len(group)>=3:
        parent_ids={str(eligible[ids.index(k)].get_subplotspec().get_gridspec()) for k in group}
        ex.append({'panels':group,'checks':['vertical-gutter'],'reason':'Column spans distinct main-panel rows and nested stacks; outer and inner gutters intentionally differ. Stack spacing remains uniform within each original nested GridSpec.'})
for group in rg:
    if len(group)>=3:
        widths=[eligible[ids.index(k)].get_position().width for k in group]
        if np.ptp(widths)>1e-4:
            ex.append({'panels':group,'checks':['panel-width','horizontal-gutter'],'reason':'Nested composites intentionally allocate widths to distinct quantities; within-product repeats are compared separately.'})
require_matplotlib_panel_alignment(fig,axes=eligible,panel_ids=ids,row_groups=rg,column_groups=cg,
    exemptions=ex,require_panel_labels=False,json_out=QA/'alignment.json',overlay_svg=QA/'alignment.svg',strict=True)
fig.savefig(OUTDIR/f'Figure{FIGURE_NUMBER}_v2.0.png',dpi=450,facecolor='white')
fig.savefig(OUTDIR/f'Figure{FIGURE_NUMBER}_v2.0.pdf',dpi=450,facecolor='white')
fig.savefig(OUTDIR/f'Figure{FIGURE_NUMBER}_v2.0.svg',dpi=450,facecolor='white')
fig.canvas.draw()
meta=[{'axis':j,'bounds':list(a.get_position().bounds),'title':a.get_title(),'xlabel':a.get_xlabel(),'ylabel':a.get_ylabel(),
       'texts':[t.get_text() for t in a.texts]} for j,a in enumerate(fig.axes)]
(QA/'axes.json').write_text(json.dumps(meta,ensure_ascii=False,indent=2),encoding='utf-8')
print(f'Figure {FIGURE_NUMBER}: exported, {len(before)} numerical artist arrays unchanged.')
plt.close(fig)
